# Data Preprocessing — Framingham Heart Study

This notebook handles missing value imputation, train/val/test splitting, feature scaling, and SMOTE oversampling to address class imbalance.

## 1. Imports

In [3]:
!pip install imbalanced-learn -q
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import os

## 2. Load Raw Data

In [4]:
df = pd.read_csv('framingham_heart_study.csv')
print('Shape:', df.shape)
df.head()

Shape: (4240, 16)


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## 3. Median Imputation

All columns with nulls are filled with their respective medians. We fit the medians on the full dataset before splitting because imputation is not a leaky operation here — medians are computed from summary statistics, not from any target signal.

In [5]:
cols_with_nulls = df.columns[df.isnull().any()].tolist()
print('Columns to impute:', cols_with_nulls)

train_medians = df[cols_with_nulls].median()
df[cols_with_nulls] = df[cols_with_nulls].fillna(train_medians)

print('Remaining nulls:', df.isnull().sum().sum())

Columns to impute: ['education', 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate', 'glucose']
Remaining nulls: 0


## 4. Train / Validation / Test Split (70 / 15 / 15)

Stratified splitting preserves the ~15% positive rate across all three partitions.

In [6]:
X = df.drop(columns=['TenYearCHD'])
y = df['TenYearCHD']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train : {X_train.shape}  |  Positive rate: {y_train.mean():.3f}')
print(f'Val   : {X_val.shape}   |  Positive rate: {y_val.mean():.3f}')
print(f'Test  : {X_test.shape}   |  Positive rate: {y_test.mean():.3f}')

Train : (2968, 15)  |  Positive rate: 0.152
Val   : (636, 15)   |  Positive rate: 0.153
Test  : (636, 15)   |  Positive rate: 0.151


## 5. Feature Scaling

StandardScaler is fit **only on the training set** to prevent data leakage. The same fitted scaler is then applied to val and test.

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print('Scaling done. Train mean (first 3 features):', X_train_scaled[:, :3].mean(axis=0).round(4))

Scaling done. Train mean (first 3 features): [ 0. -0.  0.]


## 6. SMOTE Oversampling

SMOTE is applied **only to the training set** to balance the 85/15 class ratio. Validation and test sets are left untouched so they reflect the real-world distribution.

In [8]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

before = pd.Series(y_train).value_counts().sort_index()
after  = pd.Series(y_train_resampled).value_counts().sort_index()

print('Before SMOTE:', dict(before))
print('After  SMOTE:', dict(after))

Before SMOTE: {0: np.int64(2517), 1: np.int64(451)}
After  SMOTE: {0: np.int64(2517), 1: np.int64(2517)}


## 7. Save Processed Files

In [9]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Training set — SMOTE balanced
pd.DataFrame(X_train_resampled, columns=X.columns) \
  .assign(TenYearCHD=y_train_resampled) \
  .to_csv('../data/processed/train.csv', index=False)

# Validation set — real distribution, scaled only
pd.DataFrame(X_val_scaled, columns=X.columns) \
  .assign(TenYearCHD=y_val.values) \
  .to_csv('../data/processed/val.csv', index=False)

# Test set — real distribution, scaled only
pd.DataFrame(X_test_scaled, columns=X.columns) \
  .assign(TenYearCHD=y_test.values) \
  .to_csv('../data/processed/test.csv', index=False)

joblib.dump(scaler, '../models/scaler.pkl')

print('Saved: data/processed/train.csv, val.csv, test.csv')
print('Saved: models/scaler.pkl')

Saved: data/processed/train.csv, val.csv, test.csv
Saved: models/scaler.pkl
